# combine

> Combine calibrated dithers into a single mosaic using swarp.

In [ ]:
# | default_exp euclid.combine

In [ ]:
# | exporti

import subprocess
import tempfile
from abc import ABC, abstractmethod
from datetime import datetime
from itertools import chain
from pathlib import Path
from shutil import copy2, rmtree

import numpy as np
from astropy import units as u
from astropy.io import fits
from photutils.background import Background2D

from nicl.euclid.constants import (
    MER,
    NISP,
    SWARP_CONFIG_MER,
    SWARP_CONFIG_NISP,
    SWARP_CONFIG_VIS,
    VIS,
)
from nicl.euclid.continuity import bkg_match_corr
from nicl.euclid.data_access import DataAccess
from nicl.euclid.skyflat import apply_skyflat, read_skyflat
from nicl.euclid.utilities import (
    default_data_path,
    get_dither_id_from_filename,
    get_obs_id_from_filename,
    round_up_box_size,
)
from nicl.mask import fast_mask  # noqa: F401
from nicl.utilities import (
    create_sky_rectangle,
    does_image_overlap_with_skyregion,
    parse_input_for_angular_size,
    parse_input_for_skycoord,
)

In [ ]:
# | hide
# additional imports for examples

from astropy.coordinates import SkyCoord

from nicl.utilities import physical_to_angular

In [ ]:
# | exporti
# base class for combining images


class Combiner(ABC):
    """Combines Euclid images using SWarp."""

    def __init__(
        self,
        in_dir=None,  # directory at which to find the input images
        out_dir=None,  # directory at which to place output images
        filters=None,  # list of filters to combine
        ids=None,  # list of obsids or tile ids to combine
        cutout_cen=None,  # central coordinates of the cutout
        cutout_size=None,  # size of the cutout in angular units
        name=None,  # suffix for the output file basename
        individual_dithers=False,  # if True, dithers are combined separately
        bkg_sub=True,  # to subtract background or not
        bkg_match=False,  # match background between different exposures
        bkg_mesh_size=None,  # size of the background mesh boxes in angular units
        bkg_filter_size=None,  # median filter background over `bkg_filter_size` x `bkg_filter_size` boxes
        multi_chip_bkg=False,  # if True, combine all chips/quads first for background subtraction
        autodark_corr=False,  # if True, apply dark current correction for VIS dithers
        autodark_dir=None,  # directory to find autodarks for VIS dithers
        release_name=None,  # the data release to use to find obs_ids if required, e.g., 'Q1_R1', 'DR1'
        instrument=None,  # Euclid instrument,e.g., VIS or NISP in nicl.euclid.constants
        pixel_scale=None,  # pixel scale in arcsec/pixel
        swarp_config=None,  # default SWarp configuration file content
        overwrite=False,  # overwrite existing combined image files
        debug=False,  # retain intermediate files for checking
        **kwargs,  # command line arguments to pass to SWarp, will override the defaults
    ):
        self.in_dir = in_dir
        self.out_dir = out_dir
        self.cutout_cen = cutout_cen
        self.cutout_size = cutout_size
        self.bkg_mesh_size = bkg_mesh_size
        self.instrument = instrument
        self.swarp_config = swarp_config
        self.bkg_sub = bkg_sub
        self.bkg_match = bkg_match
        self.bkg_filter_size = bkg_filter_size
        self.release_name = release_name
        self.individual_dithers = individual_dithers
        self.overwrite = overwrite
        self.debug = debug
        # broaden cutout size for bkg match correction
        if self.bkg_match and self.cutout_size is not None:
            # broaden the cutout size by the maximum dither offset
            dither_offsets = np.array(self.instrument.dither_offsets)
            delta_cutout_size = (
                2.1 * np.max(np.linalg.norm(dither_offsets, axis=1)) * u.arcsec
            )
            self.cutout_size_broad = (
                self.cutout_size[0] + delta_cutout_size,
                self.cutout_size[1] + delta_cutout_size,
            )
        else:
            self.cutout_size_broad = None
        # check ids
        if ids is None:
            # try getting ids from cutout_cen and cutout_size
            if self.cutout_cen is not None and self.cutout_size is not None:
                ids = self._get_ids()
                if len(ids) > 0:
                    self.ids = ids
                else:
                    raise ValueError(
                        "No observations found for the given cutout parameters."
                    )
            else:
                raise ValueError(
                    "Either ids or cutout_cen and cutout_size must be specified."
                )
        else:
            self.ids = np.atleast_1d(ids)
        # assemble output file name stem suffix from ids if not provided
        if name is None or self.individual_dithers:
            self.name = "-".join([str(id) for id in self.ids])
        else:
            self.name = name
        # check if filters belong to the instrument; if not specified, use all instrument filters
        if filters is None:
            if self.instrument is not None:
                self.filters = self.instrument.filters
            else:
                raise ValueError(
                    "Either filters or instrument must be specified to determine the filters."
                )
        else:
            filters = np.atleast_1d(filters)
            if self.instrument is not None:
                if set(filters).issubset(self.instrument.filters):
                    self.filters = filters
                else:
                    raise ValueError(
                        "The filters provided are not available for the instrument."
                    )
            else:
                self.filters = filters
        if pixel_scale is None:
            self.pixel_scale = self.instrument.pix_scale
        else:
            self.pixel_scale = pixel_scale
        # determine if multiple chips need to be combined first for bkg subtraction
        # override user input if the mesh size is larger than the chip/quad size
        self.multi_chip_bkg = multi_chip_bkg
        if self.bkg_sub and self.bkg_mesh_size is not None:
            rd_size = (
                np.array(self.instrument.detector_dimensions)
                * self.instrument.pix_scale
            )
            mesh_size = np.array(
                [mesh.to_value("arcsec") for mesh in self.bkg_mesh_size]
            )
            if np.any(mesh_size > rd_size):
                self.multi_chip_bkg = True
        # check autodark settings
        self.autodark_corr = autodark_corr
        if self.autodark_corr:
            if autodark_dir is None:
                raise ValueError(
                    "Autodark correction requires specifying the directory to find autodarks."
                )
            self.autodark_dir = Path(autodark_dir).expanduser()
        else:
            self.autodark_dir = None
        # assemble command line arguments to pass to SWarp
        self._swarp_extra_args = self._parse_swarp_args(kwargs)
        print(f"Initialized {self}")

    def combine(self):
        for filter in self.filters:
            self.combine_per_filter(filter)

    @abstractmethod
    def combine_per_filter(self, filter):
        pass

    def _parse_swarp_args(self, kwargs):
        kwargs = {k.upper(): v for k, v in kwargs.items()}
        swarp_args = []
        for key, value in kwargs.items():
            if key in ["CENTER", "IMAGE_SIZE"]:
                raise ValueError(
                    f"{key} should be supplied as arguments to {self.__class__.__name__}"
                )
            else:
                swarp_args.extend([f"-{key}", str(value)])
        # specify pixel scale
        if self.pixel_scale is not None:
            swarp_args.extend(
                [
                    "-PIXEL_SCALE",
                    f"{self.pixel_scale:.2f}",
                ]
            )
        # specify cutout center, cutout size will be inferred later depending on resampling or stacking
        if self.cutout_cen is not None:
            swarp_args.extend(
                [
                    "-CENTER",
                    f"{self.cutout_cen.to_string('hmsdms', sep=':', precision=2)}",
                    "-CENTER_TYPE",
                    "MANUAL",
                ]
            )
        return swarp_args

    @abstractmethod
    def _get_ids(self):
        pass

    @abstractmethod
    def _find_images(self):
        pass

    @abstractmethod
    def _prepare_input_for_swarp(self):
        pass

    def _run_swarp(self, tmpdir, resample=True, stack=True):
        with open(tmpdir / "config.swarp", "w") as f:
            f.write(self.swarp_config)
        if resample:
            input = "images.list"
        else:
            input = "resamp_images.list"
        swarp_cmd = ["swarp", f"@{input}", "-c", "config.swarp"]
        swarp_cmd.extend(self._swarp_extra_args)
        # infer cutout size in pixels
        if self.cutout_size is not None:
            # broaden cutout size for the resample run if bkg_match is True
            cutout_size_ = (
                self.cutout_size_broad
                if self.bkg_match and resample and not stack
                else self.cutout_size
            )
            cutout_size_pix = [
                round(cutsize.to(u.arcsec).value / self.pixel_scale)
                for cutsize in cutout_size_
            ]
            swarp_cmd.extend(
                ["-IMAGE_SIZE", f"{cutout_size_pix[0]},{cutout_size_pix[1]}"]
            )
        if resample:
            swarp_cmd.extend(["-RESAMPLE", "Y"])
        else:
            swarp_cmd.extend(["-RESAMPLE", "N"])
        if stack:
            swarp_cmd.extend(["-COMBINE", "Y"])
        else:
            swarp_cmd.extend(["-COMBINE", "N"])
        print(f"Running SWarp: {' '.join(swarp_cmd)}")
        start_time = datetime.now()
        try:
            run_swarp = subprocess.run(
                swarp_cmd,
                cwd=tmpdir,
                text=True,
                capture_output=True,
                check=True,
            )
            if self.debug:
                print(run_swarp.stderr)
            end_time = datetime.now()
            elapsed_secs = (end_time - start_time).total_seconds()
            elapsed_mins = elapsed_secs / 60
            print(
                f"SWarp finished successfully. Elapsed time: {elapsed_mins:.1f} mins."
            )
            return True
        except subprocess.CalledProcessError as e:
            print(
                f"Command '{' '.join(e.cmd)}' returned non-zero exit status, please check its stderr below."
            )
            print(e.stderr)
            return False

    @abstractmethod
    def _post_process(self):
        pass

In [ ]:
# | exporti
# mixin class for pre and post processing VIS and NISP dithers
# for reducing redundancy because VISCombiner and NISPCombiner are similar


class DithersMixin:
    def __str__(self):
        if self.cutout_cen is not None:
            cutout_cen_str = self.cutout_cen.to_string("hmsdms", precision=1)
        else:
            cutout_cen_str = "ALL"
        if self.cutout_size is not None:
            cutout_size_str = (
                f"{self.cutout_size[0].to_value('arcmin'):.1f}"
                + "x"
                + f"{self.cutout_size[1].to_value('arcmin'):.1f}"
                + " arcmin^2"
            )
        else:
            cutout_size_str = "AUTO"
        pixel_scale_str = f"{self.pixel_scale:.2f}"
        obj_str = f"{self.__class__.__name__}(obsids={self.ids}, filters={self.filters}, cutout_cen={cutout_cen_str}, cutout_size={cutout_size_str}, pixel_scale={pixel_scale_str}"
        if self.individual_dithers:
            obj_str += ", individual_dithers=True"
        if self.multi_chip_bkg:
            obj_str += ", multi_chip_bkg=True"
        else:
            obj_str += f", bkg_sub={self.bkg_sub}"
        if self.autodark_corr:
            obj_str += ", autodark_corr=True"
        obj_str += ")"
        return obj_str

    def __repr__(self):
        return self.__str__()

    def _get_obsids(self):
        da = DataAccess(release_name=self.release_name)
        if self.bkg_match:
            radius = 0.5 * max(*self.cutout_size_broad)
        else:
            radius = 0.5 * max(*self.cutout_size)
        ids = da.find_observations_for_target(
            self.cutout_cen.ra, self.cutout_cen.dec, radius, fully_contained=False
        )
        return ids

    def _find_dithers(self, pattern, filter):
        """Find paths to dithers for a given filter."""
        image_paths = []
        for id in np.atleast_1d(self.ids):
            image_paths.extend(
                self.in_dir.glob(pattern.format(obsid=id, filter=filter))
            )
        n_dithers = len(image_paths)
        n_obs = len(self.ids)
        # soft checks since some obsid may have more than the usual number of dithers
        print(
            f"Found {n_dithers} {filter} dithers for {n_obs} obsids. Expected {n_obs * self.instrument.n_dithers_per_obs} dithers."
        )
        return image_paths

    def _prepare_dithers_for_swarp(self, dithers, tmpdir):
        start_time = datetime.now()
        sci_fns = []
        for dither in dithers:
            # for now we throw away the short exposures
            if self.autodark_corr and self.instrument.name == "VIS":
                dither_id = get_dither_id_from_filename(dither.name)
                if dither_id.endswith("2"):
                    continue
            with fits.open(dither) as hdul:
                if self.multi_chip_bkg:
                    primary_hdr = hdul[0].header
                    sci_data = hdul[1].data
                    rms_data = hdul[2].data
                    sci_ext_hdr = hdul[1].header
                    rms_ext_hdr = hdul[2].header
                    bad_pix_mask = np.isinf(rms_data)
                    # compute weight map
                    with np.errstate(divide="ignore"):
                        np.divide(1.0, np.square(rms_data), out=rms_data)
                    # set the weight to 0 for bad pixels
                    rms_data[bad_pix_mask] = 0
                    sci_hdul = fits.HDUList(
                        [
                            fits.PrimaryHDU(header=primary_hdr),
                            fits.ImageHDU(sci_data, sci_ext_hdr),
                        ]
                    )
                    weight_hdul = fits.HDUList(
                        [
                            fits.PrimaryHDU(header=primary_hdr),
                            fits.ImageHDU(rms_data, rms_ext_hdr),
                        ]
                    )
                    sci_fn = dither.name
                    wt_fn = dither.name.replace(".fits", ".weight.fits")
                    sci_fns.append(sci_fn)
                    sci_hdul.writeto(tmpdir / sci_fn)
                    weight_hdul.writeto(tmpdir / wt_fn)
                else:
                    for ext in self.instrument.extnames:
                        sci_ext_hdr = hdul[f"{ext}.SCI"].header
                        # check if the cutout region overlaps with the chip/quad
                        if self.cutout_cen is not None and self.cutout_size is not None:
                            if self.bkg_match:
                                # broaden the cutout size for background matching
                                cutout_reg = create_sky_rectangle(
                                    center=self.cutout_cen,
                                    width=self.cutout_size_broad[0],
                                    height=self.cutout_size_broad[1],
                                )
                            else:
                                cutout_reg = create_sky_rectangle(
                                    center=self.cutout_cen,
                                    width=self.cutout_size[0],
                                    height=self.cutout_size[1],
                                )
                            if not does_image_overlap_with_skyregion(
                                sci_ext_hdr, cutout_reg
                            ):
                                continue
                        sci_data = hdul[f"{ext}.SCI"].data
                        rms_data = hdul[f"{ext}.RMS"].data
                        if self.instrument.name == "VIS":
                            dq_data = hdul[f"{ext}.FLG"].data
                        elif self.instrument.name == "NISP":
                            dq_data = hdul[f"{ext}.DQ"].data
                        primary_hdr = hdul[0].header
                        rms_ext_hdr = hdul[f"{ext}.RMS"].header
                        bad_pix_mask = np.any(
                            [
                                (dq_data & 2**bit > 0)
                                for bit in self.instrument.bad_pix_bits
                            ],
                            axis=0,
                        )
                        # perform autodark correction for VIS dithers
                        if self.autodark_corr and self.instrument.name == "VIS":
                            obsid = get_obs_id_from_filename(dither.name)
                            # TODO: separate applying autodark correction for short and long exposures
                            autodark, coarse_factor = read_skyflat(
                                obsid, "VIS", ext, self.autodark_dir
                            )
                            sci_data = apply_skyflat(
                                sci_data,
                                autodark,
                                interpolation_method="linear",
                                coarse_factor=coarse_factor,
                            )
                        if self.instrument.name == "NISP":
                            # compute FLXSCALE for swarp only for NISP
                            # VIS already has the proper FLXSCALE in the headers
                            exptime = primary_hdr["EXPTIME"]
                            photfnu = primary_hdr["PHOTFNU"]
                            phrelex = primary_hdr["PHRELEX"]
                            phreldt = sci_ext_hdr["PHRELDT"]
                            flxscale = (1.0 / exptime) * photfnu * phrelex * phreldt
                            sci_ext_hdr.set(
                                "FLXSCALE",
                                flxscale,
                                "Combined photometric scaling factors",
                            )
                        # compute weight map
                        with np.errstate(divide="ignore"):
                            np.divide(1.0, np.square(rms_data), out=rms_data)
                        # set the weight to 0 for bad pixels
                        rms_data[bad_pix_mask] = 0
                        sci_hdul = fits.HDUList(
                            [
                                fits.PrimaryHDU(header=primary_hdr),
                                fits.ImageHDU(sci_data, sci_ext_hdr),
                            ]
                        )
                        weight_hdul = fits.HDUList(
                            [
                                fits.PrimaryHDU(header=primary_hdr),
                                fits.ImageHDU(rms_data, rms_ext_hdr),
                            ]
                        )
                        sci_fn = f"{dither.stem}.{ext}.fits"
                        wt_fn = sci_fn.replace(".fits", ".weight.fits")
                        sci_fns.append(sci_fn)
                        sci_hdul.writeto(tmpdir / sci_fn)
                        weight_hdul.writeto(tmpdir / wt_fn)
        if not sci_fns:
            print("No valid images preprocessed for SWarp.")
            return False
        # prepare image list for swarp
        with open(tmpdir / "images.list", "w") as f:
            for fn in sci_fns:
                f.write(f"{fn}[1]\n")
        end_time = datetime.now()
        elapsed_secs = (end_time - start_time).total_seconds()
        elapsed_mins = elapsed_secs / 60
        print(f"Preparing science and weight images took {elapsed_mins:.1f} mins.")
        return True

    def _prepare_resampled_for_stack(self, tmpdir):
        """Background matching and/or subtraction before stacking."""
        resamp_sci_images = list(tmpdir.glob("*resamp.fits"))
        if self.bkg_match:
            start_time = datetime.now()
            resamp_imgs_not_corrected = bkg_match_corr(resamp_sci_images)
            if resamp_imgs_not_corrected:
                cutout_reg = create_sky_rectangle(
                    center=self.cutout_cen,
                    width=self.cutout_size[0],
                    height=self.cutout_size[1],
                )
                for path in resamp_imgs_not_corrected:
                    hdr = fits.getheader(
                        path.with_suffix("").with_suffix("").with_suffix(path.suffix), 1
                    )
                    if does_image_overlap_with_skyregion(hdr, cutout_reg):
                        print(f"{path} is not corrected for background matching.")
            end_time = datetime.now()
            elapsed_mins = (end_time - start_time).total_seconds() / 60
            print(f"Background matching took {elapsed_mins:.1f} mins.")
        # subtract background if requested
        if self.bkg_sub:
            start_time = datetime.now()
            for sci_image in resamp_sci_images:
                weight_image = sci_image.with_suffix(".weight.fits")
                with (
                    fits.open(sci_image, mode="update") as sci_hdul,
                    fits.open(weight_image) as weight_hdul,
                ):
                    # swarp resampled images only has one primary hdu
                    sci_img = sci_hdul[0].data
                    weight_img = weight_hdul[0].data
                    if self.bkg_mesh_size is not None:
                        bkg_mesh_size_pix = (
                            round_up_box_size(
                                sci_img.shape[0],
                                self.bkg_mesh_size[0].to(u.arcsec).value
                                / self.instrument.pix_scale,
                            ),
                            round_up_box_size(
                                sci_img.shape[1],
                                self.bkg_mesh_size[1].to(u.arcsec).value
                                / self.instrument.pix_scale,
                            ),
                        )
                    else:
                        bkg_mesh_size_pix = None
                    # this includes bad pixels and blank pixels
                    bad_pix_mask = weight_img == 0
                    # coverage mask for blank pixels
                    coverage_mask = (weight_img == 0) & (sci_img == 0)
                    # remove blank pixels from the bad pixel mask
                    bad_pix_mask[coverage_mask] = False
                    sci_img = sub_bkg(
                        sci_img,
                        dq_mask=bad_pix_mask,
                        obj_mask="fast_mask",
                        coverage_mask=coverage_mask,
                        mesh_size=bkg_mesh_size_pix,
                        filter_size=(
                            self.bkg_filter_size,
                            self.bkg_filter_size,
                        ),
                        exclude_percentile=90.0,
                    )
                    sci_hdul[0].data = sci_img
                    sci_hdul.flush()
            end_time = datetime.now()
            elapsed_mins = (end_time - start_time).total_seconds() / 60
            print(f"Background subtraction took {elapsed_mins:.1f} mins.")
        with open(tmpdir / "resamp_images.list", "w") as f:
            for fn in resamp_sci_images:
                f.write(f"{fn}\n")
        return True

    def _post_process_stack_and_weight(self, tmpdir, out_fn):
        """Clean up FITS headers and copy the output to the desired directory."""
        start_time = datetime.now()
        if self.multi_chip_bkg:
            # remove the files in the temporary input directory
            rmtree(self.in_dir)
        with (
            fits.open(tmpdir / "coadd.fits", memmap=True) as hdul_sci,
            fits.open(tmpdir / "coadd.weight.fits", memmap=True) as hdul_weights,
        ):
            # this is necessary because they are both PrimaryHDU objects
            hdu_sci = fits.ImageHDU(hdul_sci[0].data, hdul_sci[0].header)
            hdu_rms = fits.ImageHDU(hdul_weights[0].data, hdul_weights[0].header)
            # set science image to NaN where the weight is zero
            hdu_sci.data[hdu_rms.data == 0] = np.nan
            # convert the weight to RMS
            with np.errstate(divide="ignore"):
                np.divide(1.0, hdu_rms.data, out=hdu_rms.data)
            np.sqrt(hdu_rms.data, out=hdu_rms.data)
            # clean up the headers
            hdu_sci.header.set("XTENSION", "IMAGE   ", "Image extension", before=True)
            hdu_sci.header.set("PCOUNT", 0, "number of parameters", after="NAXIS2")
            hdu_sci.header.set("GCOUNT", 1, "number of groups", after="PCOUNT")
            hdu_sci.header.set("EXTNAME", "SCI", "Extension name", after="GCOUNT")
            hdu_sci.header.set("ZPAB", 23.9)
            hdu_sci.header.set("ZPABE", 0.0)
            hdu_sci.header.set("ZPVEGA", 1.0)
            hdu_sci.header.set("ZPVEGAE", 0.0)
            hdu_sci.header.remove("SIMPLE", ignore_missing=True)
            hdu_rms.header.set("XTENSION", "IMAGE   ", "Image extension", before=True)
            hdu_rms.header.set("PCOUNT", 0, "number of parameters", after="NAXIS2")
            hdu_rms.header.set("GCOUNT", 1, "number of groups", after="PCOUNT")
            hdu_rms.header.set("EXTNAME", "RMS", "Extension name", after="GCOUNT")
            for key in ("SIMPLE", "ZPAB", "ZPAB", "ZPVEGA", "ZPVEGAE"):
                hdu_rms.header.remove(key, ignore_missing=True)
            hdul = fits.HDUList([fits.PrimaryHDU(), hdu_sci, hdu_rms])
            self.out_dir.mkdir(parents=True, exist_ok=True)
            hdul.writeto(self.out_dir / out_fn, overwrite=self.overwrite)
        end_time = datetime.now()
        elapsed_secs = (end_time - start_time).total_seconds()
        print(
            f"Postprocessing took {elapsed_secs:.1f} secs. Output saved to {self.out_dir / out_fn}"
        )


def sub_bkg(
    img,  # input image
    dq_mask=None,  # data quality mask for bad pixels
    coverage_mask=None,  # mask for blank pixels
    obj_mask="fast_mask",  # object mask; can be a callable function or an array
    mesh_size=None,  # mesh size for background modeling; default to 1x1 mesh if not specified
    **kwargs,  # additional arguments for Background2D
):
    """Model background and subtract it from the input image. Optionally building an object mask if the user provides a function name instead of an array via `obj_mask`."""
    img = np.asarray(img)
    if dq_mask is not None:
        dq_mask = np.asarray(dq_mask).astype(bool)
        if dq_mask.shape != img.shape:
            raise ValueError(
                "The shape of the data quality mask does not match that of the input image."
            )
    if coverage_mask is not None:
        coverage_mask = np.asarray(coverage_mask).astype(bool)
        if coverage_mask.shape != img.shape:
            raise ValueError(
                "The shape of the coverage mask does not match that of the input image."
            )
    # check if need to build object mask
    if isinstance(obj_mask, str):
        # determine if obj_mask is a callable function
        if obj_mask in globals() and callable(globals()[obj_mask]):
            obj_mask = globals()[obj_mask]
        elif obj_mask in locals() and callable(locals()[obj_mask]):
            obj_mask = locals()[obj_mask]
        else:
            raise ValueError(f"{obj_mask} is not a valid function name")
        match obj_mask.__name__:
            case "fast_mask":
                obj_mask = obj_mask(
                    img,
                    coverage_mask=coverage_mask,
                    estimate_background=True,
                    return_threshold=False,
                )
            case _:
                obj_mask = obj_mask(img)
    else:
        obj_mask = np.asarray(obj_mask).astype(bool)
        if obj_mask.shape != img.shape:
            raise ValueError(
                "The shape of the object mask does not match that of the input image."
            )
    # combine the dq_mask and obj_mask for bkg subtraction
    if dq_mask is not None:
        mask = dq_mask | obj_mask
    else:
        mask = obj_mask
    # default 1x1 mesh
    if mesh_size is None:
        mesh_size = img.shape[0], img.shape[1]
    bg = Background2D(
        img,
        mesh_size,
        mask=mask,
        coverage_mask=coverage_mask,
        **kwargs,
    )
    return img - bg.background

In [ ]:
# | exporti
# subclass for combing NISP images


class NISPCombiner(DithersMixin, Combiner):
    """Combine NISP images using SWarp."""

    def __init__(self, **kwargs):
        super().__init__(
            instrument=NISP,
            swarp_config=SWARP_CONFIG_NISP,
            autodark_corr=False,
            **kwargs,
        )

    def combine_per_filter(self, filter):
        if self.multi_chip_bkg:
            print(
                "First combine all chips into a single image for background subtraction."
            )
            print("-" * 80)
            try:
                tmpdir = Path(tempfile.TemporaryDirectory(delete=False).name)
                for id in self.ids:
                    combiner = NISPCombiner(
                        in_dir=self.in_dir,
                        out_dir=tmpdir,
                        filters=self.filters,
                        ids=id,
                        cutout_cen=self.cutout_cen,
                        cutout_size=self.cutout_size,
                        individual_dithers=True,
                        bkg_sub=False,
                        release_name=self.release_name,
                        debug=self.debug,
                    )
                    combiner.combine()
            except:
                rmtree(tmpdir)
                raise
            self.in_dir = tmpdir
            print("-" * 80)
            print("Actually start combining the dithers now.")
        images = self._find_images(filter)
        if not images:
            return
        if self.individual_dithers:
            out_fn = "EUC_NIR_W-CAL-IMAGE_"
        else:
            out_fn = "EUC_NIR_W-STK_"
        out_fn += filter
        if self.name:
            out_fn += f"-{self.name}"
        out_fn += ".fits"
        if self.individual_dithers:
            for image in images:
                dither = get_dither_id_from_filename(image)
                self._combine_images(
                    [image], out_fn.replace(".fits", f"-{dither}.fits")
                )
        else:
            self._combine_images(images, out_fn)

    def _combine_images(self, images, out_fn):
        if (self.out_dir / out_fn).exists() and not self.overwrite:
            print(
                f"Output file {out_fn} already exists, but overwrite=False. Skipping combine."
            )
            return
        with tempfile.TemporaryDirectory(
            dir=Path("~").expanduser(), delete=(not self.debug)
        ) as tmpdir:
            tmpdir = Path(tmpdir)
            if self.debug:
                print(f"Intermediate files can be found in {tmpdir}/.")
                print("You must delete this folder manually when done.")
            if not self._prepare_input_for_swarp(images, tmpdir):
                return
            if not self._run_swarp(tmpdir, resample=True, stack=False):
                return
            if not self._prepare_resampled_for_stack(tmpdir):
                return
            if not self._run_swarp(tmpdir, resample=False, stack=True):
                return
            self._post_process(tmpdir, out_fn)

    def _get_ids(self):
        return super()._get_obsids()

    def _find_images(self, filter):
        """Return a list of paths to the NISP dithers."""
        return super()._find_dithers(
            pattern="**/EUC_NIR_W-CAL-IMAGE_{filter}-{obsid}-*.fits",
            filter=filter,
        )

    def _prepare_input_for_swarp(self, images, tmpdir):
        return super()._prepare_dithers_for_swarp(images, tmpdir)

    def _post_process(self, tmpdir, out_fn):
        super()._post_process_stack_and_weight(tmpdir, out_fn)

In [ ]:
# | exporti


class VISCombiner(DithersMixin, Combiner):
    """Combine VIS images using SWarp."""

    def __init__(self, **kwargs):
        super().__init__(instrument=VIS, swarp_config=SWARP_CONFIG_VIS, **kwargs)

    def _get_ids(self):
        return super()._get_obsids()

    def combine_per_filter(self, filter):
        if self.multi_chip_bkg:
            print(
                "First combine all quads into a single image for background subtraction."
            )
            print("-" * 80)
            try:
                tmpdir = Path(tempfile.TemporaryDirectory(delete=False).name)
                for id in self.ids:
                    combiner = VISCombiner(
                        in_dir=self.in_dir,
                        out_dir=tmpdir,
                        filters=self.filters,
                        ids=id,
                        cutout_cen=self.cutout_cen,
                        cutout_size=self.cutout_size,
                        individual_dithers=True,
                        bkg_sub=False,
                        autodark_corr=self.autodark_corr,
                        autodark_dir=self.autodark_dir,
                        release_name=self.release_name,
                        debug=self.debug,
                    )
                    combiner.combine()
            except:
                rmtree(tmpdir)
                raise
            self.in_dir = tmpdir
            # disable autodark correction for the second pass combining
            self.autodark_corr = False
            self.autodark_dir = None
            print("-" * 80)
            print("Actually start combining the dithers now.")
        images = self._find_images()
        if not images:
            return
        if self.individual_dithers:
            out_fn = "EUC_VIS_SWL-DET"
        else:
            out_fn = "EUC_VIS_SWL-STK"
        if self.name:
            out_fn += f"-{self.name}"
        out_fn += ".fits"
        if self.individual_dithers:
            for image in images:
                dither = get_dither_id_from_filename(image)
                self._combine_images(
                    [image], out_fn.replace(".fits", f"-{dither}.fits")
                )
        else:
            self._combine_images(images, out_fn)

    def _combine_images(self, images, out_fn):
        if (self.out_dir / out_fn).exists() and not self.overwrite:
            print(
                f"Output file {out_fn} already exists, but overwrite=False. Skipping combine."
            )
            return
        # the default temporary directory may not have enough space for VIS
        with tempfile.TemporaryDirectory(
            dir=Path("~").expanduser(), delete=(not self.debug)
        ) as tmpdir:
            tmpdir = Path(tmpdir)
            if self.debug:
                print(f"Intermediate files can be found in {tmpdir}/.")
                print("You must delete this folder manually when done.")
            if not self._prepare_input_for_swarp(images, tmpdir):
                return
            if not self._run_swarp(tmpdir, resample=True, stack=False):
                return
            if not self._prepare_resampled_for_stack(tmpdir):
                return
            if not self._run_swarp(tmpdir, resample=False, stack=True):
                return
            self._post_process(tmpdir, out_fn)

    def _find_images(self):
        """Return a list of paths to the VIS dithers."""
        return super()._find_dithers(
            pattern="**/EUC_VIS_SWL-DET-*{obsid}-*.fits",
            filter="I",
        )

    def _prepare_input_for_swarp(self, images, tmpdir):
        return super()._prepare_dithers_for_swarp(images, tmpdir)

    def _post_process(self, tmpdir, out_fn):
        super()._post_process_stack_and_weight(tmpdir, out_fn)

In [ ]:
# | exporti


class MerCombiner(Combiner):
    """Combine MER stacks using SWarp."""

    def __init__(self, **kwargs):
        if "add_bkg_mod" in kwargs:
            self.add_bkg_mod = kwargs["add_bkg_mod"]
            # remove it from kwargs
            kwargs.pop("add_bkg_mod")
        super().__init__(
            instrument=MER,
            swarp_config=SWARP_CONFIG_MER,
            bkg_sub=False,
            bkg_match=False,
            individual_dithers=False,
            multi_chip_bkg=False,
            autodark_corr=False,
            **kwargs,
        )

    def __str__(self):
        if self.cutout_cen is not None:
            cutout_cen_str = self.cutout_cen.to_string("hmsdms", precision=1)
        else:
            cutout_cen_str = "ALL"
        if self.cutout_size is not None:
            cutout_size_str = (
                f"{self.cutout_size[0].to_value('arcmin'):.1f}"
                + "x"
                + f"{self.cutout_size[1].to_value('arcmin'):.1f}"
                + " arcmin^2"
            )
        else:
            cutout_size_str = "AUTO"
        pixel_scale_str = f"{self.pixel_scale:.2f}"
        return f"{self.__class__.__name__}(tileids={self.ids}, filters={self.filters}, cutout_cen={cutout_cen_str}, cutout_size={cutout_size_str}, pixel_scale={pixel_scale_str}, add_bkg_mod={self.add_bkg_mod})"

    def __repr__(self):
        return self.__str__()

    def combine_per_filter(self, filter):
        images = self._find_images(filter)
        if not any(chain.from_iterable(images)):
            return
        out_fn = (
            "EUC_MER-MOSAIC-" if self.add_bkg_mod else "EUC_MER_BGSUB-MOSAIC-"
        ) + filter
        if len(self.name) > 0:
            out_fn += f"_{self.name}"
        out_fn += ".fits"
        if (self.out_dir / out_fn).exists() and not self.overwrite:
            print(
                f"Output file {out_fn} already exists, but overwrite=False. Skipping combine."
            )
            return
        with tempfile.TemporaryDirectory(delete=(not self.debug)) as tmpdir:
            tmpdir = Path(tmpdir)
            if self.debug:
                print(f"Intermediate files can be found in {tmpdir}/.")
                print("You must delete this folder manually when done.")
            self._prepare_input_for_swarp(images, tmpdir)
            self._run_swarp(tmpdir)
            self._post_process(tmpdir, out_fn)

    def _get_ids(self):
        da = DataAccess(release_name=self.release_name)
        radius = 0.5 * max(*self.cutout_size)
        ids = da.find_tiles_for_target(
            self.cutout_cen.ra, self.cutout_cen.dec, radius, fully_contained=False
        )
        return ids

    def _find_images(self, filter):
        """Return a list of tuple of paths to the MER stacks and bkg models."""
        images = []
        for tile_id in np.atleast_1d(self.ids):
            img_tpl = []
            img_tpl.extend(
                self.in_dir.glob(
                    f"**/{tile_id}/*/EUC_MER_BGSUB-MOSAIC-{filter}_TILE{tile_id}-*.fits"
                ),
            )
            if self.add_bkg_mod:
                img_tpl.extend(
                    self.in_dir.glob(
                        f"**/{tile_id}/*/EUC_MER_BGMOD*-{filter}_TILE{tile_id}-*.fits"
                    )
                )
            else:
                img_tpl.append(None)
            images.append(tuple(img_tpl))
        n_images = sum(1 for img in chain.from_iterable(images) if img is not None)
        n_tiles = len(self.ids)
        n_images_per_tile = 2 if self.add_bkg_mod else 1
        print(
            f"Found {n_images} {filter} images for {n_tiles} tileids. Expected {n_tiles * n_images_per_tile} images."
        )
        return images

    def _prepare_input_for_swarp(self, images, tmpdir):
        start_time = datetime.now()
        sci_fns = []
        for stack, bkg_mod in images:
            with fits.open(stack) as hdul:
                data = hdul[0].data
                hdr = hdul[0].header
                # I believe that all MER stacks of a given filter have the same ZP
                # So the codes below is just for safeguard purpose
                filter = hdr.get("FILTER", None)
                mag_zp = hdr.get("MAGZERO", None)
                if filter is None or mag_zp is None:
                    corr_factor = 1
                else:
                    match filter:
                        case "VIS":
                            expected_mag_zp = 24.6
                        case "NIR_Y":
                            expected_mag_zp = 29.8
                        case "NIR_J":
                            expected_mag_zp = 30.0
                        case "NIR_H":
                            expected_mag_zp = 29.9
                        case _:
                            expected_mag_zp = None
                    if mag_zp != expected_mag_zp:
                        corr_factor = 10 ** (0.4 * (expected_mag_zp - mag_zp))
                        hdr.set("MAGZERO", expected_mag_zp)
                    else:
                        corr_factor = 1
                # add bkg model back to the image if requested
                if self.add_bkg_mod:
                    with fits.open(bkg_mod) as hdul_bkg:
                        bkg_mod = hdul_bkg[0].data
                        data += bkg_mod
                # determine if writing temp fits files is needed
                if corr_factor != 1 or self.add_bkg_mod:
                    data *= corr_factor
                    sci_hdul = fits.HDUList(fits.PrimaryHDU(data=data, header=hdr))
                    # preserve the original file name for debugging
                    sci_fns.append(stack.name)
                    sci_hdul.writeto(tmpdir / stack.name)
                else:
                    # no bkg mod to add and no ZP correction needed, simply use the original file
                    sci_fns.append(str(stack))
        # write image list file for swarp
        with open(tmpdir / "images.list", "w") as f:
            for fn in sci_fns:
                f.write(f"{fn}\n")
        end_time = datetime.now()
        elapsed_secs = (end_time - start_time).total_seconds()
        elapsed_mins = elapsed_secs / 60
        print(f"Preparing MER stacks took {elapsed_mins:.1f} mins.")

    def _post_process(self, tmpdir, out_fn):
        """Copy the final stack to the desired directory."""
        start_time = datetime.now()
        self.out_dir.mkdir(parents=True, exist_ok=True)
        if not self.overwrite and (self.out_dir / out_fn).exists():
            raise FileExistsError(
                f"Output stack file {self.out_dir / out_fn} already exists."
            )
        copy2(tmpdir / "coadd.fits", self.out_dir / out_fn)
        end_time = datetime.now()
        elapsed_secs = (end_time - start_time).total_seconds()
        print(
            f"Postprocessing took {elapsed_secs:.1f} secs. Output saved to {self.out_dir / out_fn}"
        )

In [ ]:
# | export
# A high-level function to combine images; instantiates the appropriate class based on the input filters


def combine(
    in_dir=None,  # directory at which to find the input images
    out_dir=None,  # directory at which to place output images
    filters=None,  # list of filters to combine
    obs_ids=None,  # list of obsids to combine
    tile_ids=None,  # list of tile ids to combine
    individual_dithers=False,  # combine each dither separately
    cutout_cen=None,  # central coordinates of the cutout
    cutout_size=None,  # size of the cutout in angular units
    name=None,  # suffix for the output file basename.
    pixel_scale=None,  # pixel scale of the output image in arcsec/pix
    bkg_sub=True,  # to subtract background or not; not applicable to MER
    bkg_match=False,  # to match background or not; not applicable to MER
    bkg_mesh_size=None,  # size of the background mesh boxes in angular units
    add_bkg_mod=False,  # add back background model for MER stacks
    bkg_filter_size=3,  # median filter background over `filter_size` x `filter_size` boxes
    multi_chip_bkg=False,  # combine multiple chips before background modeling and subtraction
    autodark_corr=False,  # apply autodark correction for VIS dithers
    autodark_dir=None,  # directory where autodark files are located
    release_name="Q1_R1",  # the data release name, e.g., "Q1_R1", "DR1"
    overwrite=False,  # overwrite existing combined image files
    debug=False,  # retain intermediate files for checking and more verbose output
    dry_run=False,  # instantiate the classes without actually combining the images
    **kwargs,  # command line arguments to pass to SWarp
):
    """Combine a variety of Euclid data products to produce mosaics.

    Combine Euclid calibrated dithers from different instruments (VIS or NISP) or
    MER stacks using SWarp. Input images are found based on input ids (obs_ids
    for VIS/NISP or tile_ids for MER) or cutout parameters (center and size).
    Before running SWarp, the program optionally performs autodark correction,
    background subtraction for VIS/NISP dithers and adds background models back to
    MER mosaics, then writes the resulting images to a temporary directory where
    SWarp will run. Finally, the combined image is copied to the output directory
    with the specified name. Users can specify center, size, and pixel scale of the
    combined image.

    Parameters
    ----------
    in_dir : str, optional
        Directory where the input images are located. If not specified, attempts
        to infer this directory based on the data release name.
    out_dir : str, optional
        Directory where the output images are saved. If not specified, attempts
        to infer this directory based on the data release name.
    filters : list of str, optional
        List of filters to combine. Default is ["I", "Y", "J", "H", "VIS",
        "NIR-Y", "NIR-J", "NIR-H"]. I for VIS, [Y, J, H] for NISP, and
        [VIS, NIR-Y, NIR-J, NIR-H] for MER.
    obs_ids : list of str, optional
        List of observation IDs to be processed (for VIS or NISP data).
    tile_ids : list of str, optional
        List of tile IDs to be processed (for MER data).
    individual_dithers : bool, optional
        If True, each dither of the obsids is combined separately (only applies
        to VIS/NISP data).
    cutout_cen : str or SkyCoord object, optional
        Sky coordinates of the center of the cutout region.
    cutout_size : str, int, float, U.Quantity object, or a list/tuple/ndarray
        of them, optional
        Size of the cutout region in angular units (two values for different
        width and height).
    name : str, optional
        Suffix for the output file basename. Overriden by obsid if
        individual_dithers=True.
    pixel_scale : float, optional
        Pixel scale of the output image in arcsec/pix. If not specified, uses the
        default pixel scale of the instrument.
    bkg_sub : bool, optional
        If True, subtract background from the dithers (only for VIS/NISP data).
    bkg_match: bool, optional
        If True, match background between the dithers (only for VIS/NISP data).
        Not applicable when individual_dithers=True.
    bkg_mesh_size : str, int, float, U.Quantity object, or a list/tuple/ndarray
        of them, optional
        Angular size of background mesh boxes (only for VIS/NISP data).
    add_bkg_mod : bool, optional
        If True, add back the background models for MER mosaics.
    bkg_filter_size : int, optional
        Size of the filtering kernel for median-filtering the background model
        (only for VIS/NISP dithers).
    multi_chip_bkg : bool, optional
        If True, combine multiple chips/quads before background modeling and
        subtraction (only for VIS/NISP data). If bkg_mesh_size is larger than the
        chip/quad size, multi_chip_bkg is automatically set to True regardless of
        the user input.
    autodark_corr : bool, optional
        If True, apply autodark correction for VIS dithers.
    autodark_dir : str, optional
        Directory to search for the autodark FITS files.
    release_name : str, optional
        Data release identifier to look up default directories if in_dir or
        out_dir is not specified.
    overwrite : bool, optional
        If True, overwrite existing output image files.
    debug : bool, optional
        If True, retain intermediate files for debugging or inspection and
        print SWarp stderr.
    dry_run : bool, optional
        If True, configure the combiners without actually performing the image
        combination.
    **kwargs
        Additional keyword arguments to pass to the underlying SWarp calls.
        Note that they will override the default SWarp configuration.

    """
    # check IO designations
    if in_dir is None:
        if release_name is not None:
            in_dir = default_data_path(release_name)
        else:
            raise ValueError(
                "Either in_dir or release_name must be specified to find the input images."
            )
    else:
        in_dir = Path(in_dir).expanduser()
        if not in_dir.is_dir() or not in_dir.exists():
            raise ValueError("in_dir does not exist or is not a directory.")

    if out_dir is None:
        if release_name is not None:
            out_dir = default_data_path(f"{release_name}_clusters_test")
        else:
            raise ValueError(
                "Either out_dir or release_name must be specified to infer where to place the output images."
            )
    else:
        out_dir = Path(out_dir).expanduser()
        if out_dir.exists() and out_dir.is_file():
            raise ValueError("The output directory points to a file.")

    # check input ids
    if "ids" in kwargs or "id" in kwargs:
        raise ValueError("Please use obs_ids or tile_ids to pass the ids.")

    if individual_dithers and obs_ids is None:
        raise ValueError("obs_ids must be specified to combine individual dithers.")

    if cutout_cen is not None:
        cutout_cen = parse_input_for_skycoord(cutout_cen)

    if cutout_size is not None:
        cutout_size = tuple(parse_input_for_angular_size(cutout_size, duplicate=True))
        if len(cutout_size) != 2:
            raise ValueError("cutout_size must be a tuple of two angular sizes.")

    if bkg_mesh_size is not None:
        bkg_mesh_size = tuple(
            parse_input_for_angular_size(bkg_mesh_size, duplicate=True)
        )
        if len(bkg_mesh_size) != 2:
            raise ValueError("bkg_mesh_size must be a tuple of two angular sizes.")

    # if no filters are specified, assume that you need all filters;
    # Combiners treat None as all filters
    if filters is None:
        vis_filters = None
        nisp_filters = None
        mer_filters = None
    else:
        vis_filters = []
        nisp_filters = []
        mer_filters = []
        for filter in np.atleast_1d(filters):
            if filter in VIS.filters:
                vis_filters.append(filter)
                continue
            if filter in NISP.filters:
                nisp_filters.append(filter)
                continue
            if filter in MER.filters:
                mer_filters.append(filter)
                continue
    if vis_filters is None or vis_filters:
        # if requested to combine individual dithers, combine them separately for each obsid
        # otherwise naming output files would become an issue
        if individual_dithers:
            for obs_id in np.atleast_1d(obs_ids):
                vis_combiner = VISCombiner(
                    in_dir=in_dir,
                    out_dir=out_dir,
                    filters=vis_filters,
                    ids=obs_id,
                    individual_dithers=individual_dithers,
                    cutout_cen=cutout_cen,
                    cutout_size=cutout_size,
                    name=name,
                    pixel_scale=pixel_scale,
                    bkg_sub=bkg_sub,
                    bkg_match=False,
                    bkg_mesh_size=bkg_mesh_size,
                    bkg_filter_size=bkg_filter_size,
                    multi_chip_bkg=multi_chip_bkg,
                    autodark_corr=autodark_corr,
                    autodark_dir=autodark_dir,
                    release_name=release_name,
                    overwrite=overwrite,
                    debug=debug,
                    **kwargs,
                )
                if not dry_run:
                    try:
                        vis_combiner.combine()
                    except ValueError as e:
                        print(e)
        else:
            vis_combiner = VISCombiner(
                in_dir=in_dir,
                out_dir=out_dir,
                filters=vis_filters,
                ids=obs_ids,
                individual_dithers=individual_dithers,
                cutout_cen=cutout_cen,
                cutout_size=cutout_size,
                name=name,
                pixel_scale=pixel_scale,
                bkg_sub=bkg_sub,
                bkg_match=bkg_match,
                bkg_mesh_size=bkg_mesh_size,
                bkg_filter_size=bkg_filter_size,
                multi_chip_bkg=multi_chip_bkg,
                autodark_corr=autodark_corr,
                autodark_dir=autodark_dir,
                release_name=release_name,
                overwrite=overwrite,
                debug=debug,
                **kwargs,
            )
            if not dry_run:
                try:
                    vis_combiner.combine()
                except ValueError as e:
                    print(e)
    if nisp_filters is None or nisp_filters:
        if individual_dithers:
            for obs_id in np.atleast_1d(obs_ids):
                nisp_combiner = NISPCombiner(
                    in_dir=in_dir,
                    out_dir=out_dir,
                    filters=nisp_filters,
                    ids=obs_id,
                    individual_dithers=individual_dithers,
                    cutout_cen=cutout_cen,
                    cutout_size=cutout_size,
                    name=name,
                    pixel_scale=pixel_scale,
                    bkg_sub=bkg_sub,
                    bkg_match=False,
                    bkg_mesh_size=bkg_mesh_size,
                    bkg_filter_size=bkg_filter_size,
                    multi_chip_bkg=multi_chip_bkg,
                    release_name=release_name,
                    overwrite=overwrite,
                    debug=debug,
                    **kwargs,
                )
                if not dry_run:
                    try:
                        nisp_combiner.combine()
                    except ValueError as e:
                        print(e)
        else:
            nisp_combiner = NISPCombiner(
                in_dir=in_dir,
                out_dir=out_dir,
                filters=nisp_filters,
                ids=obs_ids,
                individual_dithers=individual_dithers,
                cutout_cen=cutout_cen,
                cutout_size=cutout_size,
                pixel_scale=pixel_scale,
                name=name,
                bkg_sub=bkg_sub,
                bkg_match=bkg_match,
                bkg_mesh_size=bkg_mesh_size,
                bkg_filter_size=bkg_filter_size,
                multi_chip_bkg=multi_chip_bkg,
                release_name=release_name,
                overwrite=overwrite,
                debug=debug,
                **kwargs,
            )
            if not dry_run:
                try:
                    nisp_combiner.combine()
                except ValueError as e:
                    print(e)
    if mer_filters is None or mer_filters:
        mer_combiner = MerCombiner(
            in_dir=in_dir,
            out_dir=out_dir,
            filters=mer_filters,
            ids=tile_ids,
            cutout_cen=cutout_cen,
            cutout_size=cutout_size,
            add_bkg_mod=add_bkg_mod,
            name=name,
            pixel_scale=pixel_scale,
            release_name=release_name,
            overwrite=overwrite,
            debug=debug,
            **kwargs,
        )
        if not dry_run:
            try:
                mer_combiner.combine()
            except ValueError as e:
                print(e)

## Examples

### Produce a H-band stack for a single Q1 NISP observation

Users should use the `combine` function instead of instantiating the various `Combiner` classes directly. Background subtraction is enabled by default.

In [ ]:
in_dir = default_data_path("Q1_R1")
out_dir = default_data_path("Q1_R1_processed_test")
id = 2682

In [ ]:
combine(in_dir=in_dir, out_dir=out_dir, obs_ids=id, filters="H", overwrite=True)

Initialized NISPCombiner(obsids=[2682], filters=['H'], cutout_cen=ALL, cutout_size=AUTO, pixel_scale=0.30 arcsec/pix, bkg_sub=True)
Found 4 H dithers for 1 obsids. Expected 4 dithers.
Preparing science and weight images took 0.4 mins.
Running SWarp: swarp @images.list -c config.swarp -RESAMPLE Y -COMBINE N
SWarp finished successfully. Elapsed time: 0.8 mins.
Background subtraction took 6.3 mins.
Running SWarp: swarp @resamp_images.list -c config.swarp -RESAMPLE N -COMBINE Y
SWarp finished successfully. Elapsed time: 0.4 mins.
Postprocessing took 4.9 secs. Output saved to /home/ppzhg/euclid_data/Q1_R1_processed_test/EUC_NIR_W-STK_H-2682.fits


### Combine chips for individual dithers in H band for a single Q1 observations

Useful for assembling chips into a single focal plane mosaic.

In [ ]:
combine(
    in_dir=in_dir,
    out_dir=out_dir,
    obs_ids=id,
    filters="H",
    individual_dithers=True,
    bkg_sub=False,
    overwrite=True,
)

Initialized NISPCombiner(obsids=[2682], filters=['H'], cutout_cen=ALL, cutout_size=AUTO, pixel_scale=0.30 arcsec/pix, individual_dithers=True, bkg_sub=False)
Found 4 H dithers for 1 obsids. Expected 4 dithers.
Preparing science and weight images took 0.1 mins.
Running SWarp: swarp @images.list -c config.swarp
SWarp finished successfully. Elapsed time: 0.2 mins.
Postprocessing took 2.9 secs. Output saved to /home/ppzhg/euclid_data/Q1_R1_processed_test/EUC_NIR_W-CAL-IMAGE_H-2682-3.fits
Preparing science and weight images took 0.1 mins.
Running SWarp: swarp @images.list -c config.swarp
SWarp finished successfully. Elapsed time: 0.2 mins.
Postprocessing took 2.6 secs. Output saved to /home/ppzhg/euclid_data/Q1_R1_processed_test/EUC_NIR_W-CAL-IMAGE_H-2682-2.fits
Preparing science and weight images took 0.0 mins.
Running SWarp: swarp @images.list -c config.swarp
SWarp finished successfully. Elapsed time: 0.2 mins.
Postprocessing took 2.6 secs. Output saved to /home/ppzhg/euclid_data/Q1_R1_pr

### Produce stacks of dithers and cutouts from MER stacks for a galaxy cluster using Euclid Q1 data

Optionally the background model can be added back to the MER stacks by setting `add_bkg_mod=True` in the `combine` function. Also note that background subtraction is not performed for MER stacks even if `bkg_sub=True`.

Note that `combine` reply on `filters` in distinguish whether the user wants to combine dithers or MER stacks. For combing dithers in VIS or NISP, `filters` should be one or more of `['I', 'Y', 'J', 'H']`; for MER stacks, `filters` should be one or more of `['VIS', 'NIR-Y', 'NIR-J', 'NIR-H']`. The filters can be mixed, e.g., `filters=['I', 'H', 'VIS', 'NIR-Y']`.

The user can also specify a pixel scale different from the native pixel scale of the input images for the output image. This can be done by passing `pixel_scale=your_choice` (case insensitive) to `combine`. The parameter should be in arcsec per pixel.

In addition, if the user wants to pass additional arguments to the `SWarp` command, they can do so by passing them as extra keyword arguments to `combine`. They will then be passed to `SWarp` as command-line arguments during runtime (i.e. "-{key.upper()} {value}"). However, users shoud not pass cutout related parameters (i.e., `CENTER` and `IMAGE_SIZE`) to `combine` as extra keyword arguments, since there are already dedicated keyword arguments that starts with `cutout`.


In [ ]:
cluster_id = "MCXCJ1754.6+6803"
z = 0.07
ra = 268.662 * u.deg
dec = 68.058 * u.deg
cutout_radius_physical = 500 * u.kpc
cutout_radius_angular = physical_to_angular(cutout_radius_physical, z)
cutout_size = 4 * cutout_radius_angular
bkg_mesh_size = 0.5 * cutout_radius_angular

In [ ]:
# here we use original data from the archive an save to a test location
in_dir = default_data_path("Q1_R1")
out_dir = default_data_path("Q1_R1_clusters_test", cluster_id)

# to use processed data of a particular version and processing stage
# in_dir = default_data_path("Q1_R1_processed_v0.2", "persistence")
# out_dir = default_data_path("Q1_R1_clusters_v0.2", cluster_id)

In [ ]:
combine(
    in_dir=in_dir,
    out_dir=out_dir,
    bkg_mesh_size=bkg_mesh_size,
    add_bkg_mod=True,
    cutout_cen=SkyCoord(ra, dec),
    cutout_size=cutout_size,
    name=cluster_id,
    filters=["H", "VIS", "NIR-H"],
    pixel_scale=0.3,
    overwrite=True,
)

INFO: OK [astroquery.utils.tap.core]
Initialized NISPCombiner(obsids=[2683 2685], filters=['H'], cutout_cen=17h54m38.9s +68d03m28.8s, cutout_size=24.9x24.9 arcmin^2, pixel_scale=0.30 arcsec/pix, bkg_sub=True)
Found 8 H dithers for 2 obsids. Expected 8 dithers.
Preparing science and weight images took 1.3 mins.
Running SWarp: swarp @images.list -c config.swarp -PIXEL_SCALE 0.3 -IMAGE_SIZE 4988,4988 -CENTER 17:54:38.88 +68:03:28.80 -CENTER_TYPE MANUAL
SWarp finished successfully. Elapsed time: 0.3 mins.
Postprocessing took 0.3 secs. Output saved to /home/ppzhg/euclid_data/Q1_R1_clusters_test/MCXCJ1754.6+6803/EUC_NIR_W-STK_H-MCXCJ1754.6+6803.fits
INFO: OK [astroquery.utils.tap.core]
Initialized MerCombiner(tileids=[102160335 102160336 102160607 102160608], filters=['VIS' 'NIR-H'], cutout_cen=17h54m38.9s +68d03m28.8s, cutout_size=24.9x24.9 arcmin^2, pixel_scale=0.30 arcsec/pix, add_bkg_mod=True)
Found 8 VIS images for 4 tileids. Expected 8 images.
Preparing MER stacks took 0.4 mins.
Runnin

By default the sky background modeling and subtraction is performed within each chip/quad. This sometimes leads to over subtraction when there are extended objects filling up a good portion of the chip/quad. In such cases, the user can choose to perform the sky background modeling and subtraction over the entire FoV by setting `multi_chip_bkg=True` in the `combine` function. The program will first combine all the chips/quads into a single image and then perform the sky background modeling and subtraction, which unfortunately takes longer time to process. But, it is especially useful for VIS images because a VIS quad is much smaller than a NISP chip.

In [ ]:
combine(
    in_dir=in_dir,
    out_dir=out_dir,
    bkg_mesh_size=bkg_mesh_size,
    multi_chip_bkg=True,
    cutout_cen=SkyCoord(ra, dec),
    cutout_size=cutout_size,
    name=cluster_id,
    filters=["H"],
    pixel_scale=0.3,
    overwrite=True,
)

INFO: OK [astroquery.utils.tap.core]
Initialized NISPCombiner(obsids=[2683 2685], filters=['H'], cutout_cen=17h54m38.9s +68d03m28.8s, cutout_size=24.9x24.9 arcmin^2, pixel_scale=0.30 arcsec/pix, multi_chip_bkg=True)
First combine all chips into a single image for background subtraction.
--------------------------------------------------------------------------------
Initialized NISPCombiner(obsids=[2683], filters=['H'], cutout_cen=17h54m38.9s +68d03m28.8s, cutout_size=24.9x24.9 arcmin^2, pixel_scale=0.30 arcsec/pix, individual_dithers=True, bkg_sub=False)
Found 4 H dithers for 1 obsids. Expected 4 dithers.
Preparing science and weight images took 0.0 mins.
Running SWarp: swarp @images.list -c config.swarp -IMAGE_SIZE 4988,4988 -CENTER 17:54:38.88 +68:03:28.80 -CENTER_TYPE MANUAL
SWarp finished successfully. Elapsed time: 0.0 mins.
Postprocessing took 0.4 secs. Output saved to /tmp/tmp15t8wnit/EUC_NIR_W-CAL-IMAGE_H-2683-0.fits
Preparing science and weight images took 0.0 mins.
Running S

When combining VIS dithers, the user may turn on autodark correction by setting `autodark_corr=True` and providing the directory where the autodark files are located via `autodark_dir`. 

In [ ]:
combine(
    in_dir=in_dir,
    out_dir=out_dir,
    autodark_corr=True,
    autodark_dir=default_data_path("Q1_R1_processed_v0.5", "skyflat"),
    bkg_sub=False,
    cutout_cen=SkyCoord(ra, dec),
    cutout_size=cutout_size,
    name=cluster_id,
    filters=["I"],
    pixel_scale=0.3,
    overwrite=True,
)

INFO: OK [astroquery.utils.tap.core]
Initialized VISCombiner(obsids=[2683 2685], filters=['I'], cutout_cen=17h54m38.9s +68d03m28.8s, cutout_size=24.9x24.9 arcmin^2, pixel_scale=0.30 arcsec/pix, bkg_sub=False, autodark_corr=True)
Found 12 I dithers for 2 obsids. Expected 12 dithers.
Preparing science and weight images took 9.8 mins.
Running SWarp: swarp @images.list -c config.swarp -PIXEL_SCALE 0.3 -IMAGE_SIZE 4988,4988 -CENTER 17:54:38.88 +68:03:28.80 -CENTER_TYPE MANUAL
SWarp finished successfully. Elapsed time: 1.2 mins.
Postprocessing took 0.6 secs. Output saved to /home/ppzhg/euclid_data/Q1_R1_clusters_test/MCXCJ1754.6+6803/EUC_VIS_SWL-STK-MCXCJ1754.6+6803.fits


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()